# 🎯 [OPS02_M09] AAR 일지 기반 항적 왜곡/분할(Track Split) 오류 분석 및 융합 임계값 최적화
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS02/ops02_m09_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS02_M09 |
| **미션 주제** | AAR 일지 기반 항적 왜곡/분할(Track Split) 오류 분석 및 융합 임계값 최적화 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
미국-이란 간 무력 충돌 사례에서는 드론·미사일이 여러 방향에서 동시에 들어오면서 레이더, 광학(EO/IR), 음향 센서가 같은 비행체를 각자 경보해 상황판에 중복 항적이 쏟아지고, 이 때문에 위협 수를 잘못 세거나 대응이 늦어지는 문제가 주목받았다. 본 프로젝트에서는 가상 방어 지역에 배치한 가상 센서 4기와 가상 비행체 궤적으로 교육용 경보 데이터를 코드로 만들어, 연동 규격과 통합 스키마 설계, 시각 동기화와 ENU 좌표 변환 전처리, 최단거리 매칭 Baseline, 확장 칼만 필터(EKF) 항적 추적·통합, 융합 신뢰도 지수, 전술 상황판(COP), 조작관 HITL 검토, 통신 단절·재밍 스트레스 시험, AAR 재보정, 최종 통합까지 다중 센서 중복 항적 통합 시뮬레이터를 단계별로 구현한다. 최종 목표는 비행체 하나를 하나의 항적으로 정확히 보여 주어 지휘관의 인지 부하를 줄이고, 항적 통합·분리 같은 판단은 조작관이 근거와 함께 확정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/ops02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 승인 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/ops02_data_spec.md`, `docs/ops02_hitl_guide.md`, `docs/ops02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/ops02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/ops02_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/ops02_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 일지 기반 항적 왜곡/분할(Track Split) 오류 분석 및 융합 임계값 최적화
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 일지를 분석하여 항적이 둘로 쪼개지는 트랙 분할(Track Split) 및 거짓 항적 오류 사례를 식별하고 융합 임계 파라미터를 재보정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **초기 배치 파라미터의 모의 AAR 일지에서 항적 분할·거짓 항적 사례를 분석하고 융합 임계값을 재보정한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **AAR 일지의 오류를 분할, 거짓 항적, 병합 오류로 분류해 조건별로 집계 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/ops02_aar_report.md 및 configs/ops02_tuned_params.yaml` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 오류 원인 규명의 객관성 및 개선 조치의 논리적 타당성

#### 미션 PASS 조건

- 정량 기준: AAR 피드백 적용 후 종합 정확도 10%p 이상 향상, 재발 오류 0건


- 진행 기준: AAR 보고서 작성 및 파라미터 튜닝 전후 회귀 테스트 통과


- 검증 방법: 회귀 테스트(Regression Test) 스위트 전체 자동 실행 및 비교


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""초기 배치 파라미터로 운용한 모의 AAR 일지에서 항적 분할(Track Split)·거짓 항적·병합 오류 사례를 찾아 원인을 분석하고, 프로세스 잡음·게이트·확정 규칙·삭제 기준·중복 통합 게이트를 재탐색해 융합 임계값을 재보정한 뒤 새 시나리오 회귀 시험으로 정확도 향상과 재발 오류를 확인한다."""
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

# ---------------------------------------------------------------------------
# [교육용가상자료] 다중 센서(레이더·EO/IR·음향) 경보 생성기 (실제 장비 제원·배치와 무관)
# 가상 방어 지역(원점 기준 동/북 km)에 비행체 궤적을 만들고, 센서마다 '자기 위치 기준
# 거리·방위'와 '자기 시계 기준 시각'으로 경보를 낸다. 같은 비행체가 여러 센서에 동시에
# 잡히므로 경보가 중복되고, 센서마다 시계 오차·측정 오차·오경보가 다르다.
# ---------------------------------------------------------------------------
SENSORS = {
    "RDR-A": {"type": "radar", "pos": (0.0, 0.0), "sr": 0.08, "saz": 0.4, "pd": 0.95, "fa": 1.0, "max_r": 40.0, "clock": 0.0},
    "RDR-B": {"type": "radar", "pos": (16.0, -6.0), "sr": 0.10, "saz": 0.5, "pd": 0.93, "fa": 1.0, "max_r": 40.0, "clock": 0.42},
    "EOIR-C": {"type": "eoir", "pos": (5.0, 9.0), "sr": 0.40, "saz": 0.15, "pd": 0.88, "fa": 0.5, "max_r": 18.0, "clock": -0.65},
    "ACU-D": {"type": "acoustic", "pos": (10.0, 2.0), "sr": 0.60, "saz": 2.0, "pd": 0.80, "fa": 0.6, "max_r": 9.0, "clock": 0.93},
}
CENTER = np.array([8.0, 2.0])                         # 가상 방어 지역 중심
KINDS = {"uav": (0.03, 0.06), "heli": (0.04, 0.07), "fast": (0.15, 0.20)}   # 속도 범위(km/s, 추상값)
WEATHER = {"clear": {"eoir": 1.0, "acoustic": 1.0, "radar": 1.0, "fa": 1.0},
           "night": {"eoir": 0.9, "acoustic": 1.0, "radar": 1.0, "fa": 1.1},
           "fog": {"eoir": 0.55, "acoustic": 1.0, "radar": 1.0, "fa": 1.2},
           "rain": {"eoir": 0.7, "acoustic": 0.6, "radar": 0.97, "fa": 1.6}}


def _targets(rng, n, T, special):
    # [TODO : 학생 실습 9] AAR 일지 기반 항적 왜곡/분할(Track Split) 오류 분석 및 융합 임계값 최적화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_scenario(seed, n=6, T=60, weather="clear", fa_mult=1.0, dropout=None, ghost=False, special=None,
                  noise_mult=None, drift=0.0):
    """시나리오 1개: 정답 궤적, 센서 경보(보고) 목록, 시계 동기화용 기준 신호(heartbeat)를 만든다.
    dropout=(센서, 시작, 끝): 통신 단절, ghost: 한 센서에만 보이는 거짓 항적(기만·스푸핑),
    noise_mult={센서: 배수}: 재밍으로 측정 잡음 증가, drift: 음향 센서 시계가 초당 drift초씩 흐트러짐."""
    rng = np.random.default_rng(seed)
    targets, kinds = _targets(rng, n, T, special)
    w = WEATHER[weather]
    reports, hb, mid = [], [], 0
    gpos = CENTER + np.array([-9.0, 7.0])
    for t in range(T):
        for sid, s in SENSORS.items():
            clock = s["clock"] + (drift * t if sid == "ACU-D" else 0.0)
            if t % 10 == 0:
                hb.append({"sensor": sid, "t_ref": float(t), "t_sensor": t + clock + rng.normal(0, 0.02)})
            if dropout and sid == dropout[0] and dropout[1] <= t < dropout[2]:
                continue
            nm = (noise_mult or {}).get(sid, 1.0)
            objs = [(tid, tr[t, :2]) for tid, tr in targets.items()]
            if ghost and sid == "EOIR-C":
                objs.append(("GHOST", gpos + 0.12 * t * np.array([0.6, -0.8])))
            for tid, xy in objs:
                d = xy - np.array(s["pos"])
                r = float(np.hypot(*d))
                if r > s["max_r"] or rng.random() > s["pd"] * w[s["type"]]:
                    continue
                reports.append({"msg_id": f"M{mid:05d}", "sensor": sid, "stype": s["type"],
                                "t_sensor": round(t + clock + rng.normal(0, 0.02), 3),
                                "range_km": round(r + rng.normal(0, s["sr"] * nm), 3),
                                "az_deg": round(float((np.degrees(np.arctan2(d[0], d[1])) + rng.normal(0, s["saz"] * nm)) % 360), 3),
                                "sr": s["sr"] * nm, "saz": s["saz"] * nm, "truth": tid})
                mid += 1
            for _ in range(rng.poisson(s["fa"] * fa_mult * w["fa"] * (3 if nm > 1 else 1))):   # 오경보(클러터·반사)
                reports.append({"msg_id": f"M{mid:05d}", "sensor": sid, "stype": s["type"],
                                "t_sensor": round(t + clock + rng.normal(0, 0.02), 3),
                                "range_km": round(rng.uniform(0.5, s["max_r"]), 3), "az_deg": round(rng.uniform(0, 360), 3),
                                "sr": s["sr"] * nm, "saz": s["saz"] * nm, "truth": "FA"})
                mid += 1
    return {"targets": targets, "kinds": kinds, "reports": reports, "heartbeats": hb, "T": T, "weather": weather}
# ---------------------------------------------------------------------------

# ---- 시공간 전처리 공통 함수: 정제 -> 시각 동기화 -> 공통 좌표(ENU, km) 변환 ----
def estimate_offsets(heartbeats):
    """센서별 시계 오차 = 기준 신호(heartbeat) 대비 센서 시각 차이의 중앙값(잡음에 강함)."""
    # [TODO : 학생 실습 9] AAR 일지 기반 항적 왜곡/분할(Track Split) 오류 분석 및 융합 임계값 최적화 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def clean(reports):
    """중복 메시지·결측·범위 밖(반사파 등 최대 탐지거리 초과) 경보를 걸러낸다. (정제 목록, 사유별 건수)"""
    seen, out, why = set(), [], {"중복": 0, "결측": 0, "범위 밖": 0}
    for r in reports:
        if r["msg_id"] in seen:
            why["중복"] += 1; continue
        seen.add(r["msg_id"])
        if any(r.get(k) is None or (isinstance(r.get(k), float) and np.isnan(r[k])) for k in ("t_sensor", "range_km", "az_deg")):
            why["결측"] += 1; continue
        if not 0 <= r["range_km"] <= SENSORS[r["sensor"]]["max_r"] * 1.1:
            why["범위 밖"] += 1; continue
        out.append(r)
    return out, why


def to_enu(r, offsets):
    """센서 국소 극좌표(거리·방위) -> 공통 ENU(동·북 km), 센서 시각 -> 기준 시각."""
    s, a = SENSORS[r["sensor"]], np.radians(r["az_deg"])
    return {**r, "t": r["t_sensor"] - offsets.get(r["sensor"], 0.0),
            "x": s["pos"][0] + r["range_km"] * np.sin(a), "y": s["pos"][1] + r["range_km"] * np.cos(a)}


def meas_cov(r):
    """거리·방위 오차를 ENU 공분산(2x2)으로 변환한다(1차 근사)."""
    a, rr = np.radians(r["az_deg"]), r["range_km"]
    J = np.array([[np.sin(a), rr * np.cos(a)], [np.cos(a), -rr * np.sin(a)]])
    return J @ np.diag([r["sr"] ** 2, np.radians(r["saz"]) ** 2]) @ J.T


def prepare(scn, sync=True):
    reps, _ = clean(scn["reports"])
    off = estimate_offsets(scn["heartbeats"]) if sync else {}
    return sorted((to_enu(r, off) for r in reps), key=lambda r: r["t"])

# ---- 확장 칼만 필터(EKF) 다중 센서 항적 추적기 공통 함수 ----
# 상태 x = [동(km), 북(km), 동속(km/s), 북속(km/s)], 측정 z = 센서 기준 [거리(km), 방위(rad)]
def ekf_predict(x, P, dt, q):
    F = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], float)
    G = np.array([[dt ** 2 / 2, 0], [0, dt ** 2 / 2], [dt, 0], [0, dt]])
    return F @ x, F @ P @ F.T + q * G @ G.T


def h_polar(x, spos):
    dx, dy = x[0] - spos[0], x[1] - spos[1]
    r2 = max(dx * dx + dy * dy, 1e-9); r = np.sqrt(r2)
    return np.array([r, np.arctan2(dx, dy)]), np.array([[dx / r, dy / r, 0, 0], [dy / r2, -dx / r2, 0, 0]])


def innovation(x, P, rep):
    s = SENSORS[rep["sensor"]]
    zp, H = h_polar(x, s["pos"])
    y = np.array([rep["range_km"], np.radians(rep["az_deg"])]) - zp
    y[1] = (y[1] + np.pi) % (2 * np.pi) - np.pi                      # 방위 차이는 -pi~pi로 감싼다
    S = H @ P @ H.T + np.diag([rep["sr"] ** 2, np.radians(rep["saz"]) ** 2])
    return y, H, S, float(y @ np.linalg.solve(S, y))


class Tracker:
    """센서 경보를 스캔(1초) 단위로 받아 EKF 항적을 갱신한다. 같은 스캔 안에서 센서별로
    순차 갱신하므로, 여러 센서가 같은 비행체를 보면 하나의 항적으로 통합된다."""

    def __init__(self, gate=9.21, q=4e-4, confirm=(3, 5), max_miss=4, merge_gate=6.0):
        self.gate, self.q, self.confirm, self.max_miss, self.merge_gate = gate, q, confirm, max_miss, merge_gate
        self.tracks, self.dead, self.nid, self.n_merged = [], [], 0, 0

    def _new(self, rep, k):
        C = meas_cov(rep)
        P = np.zeros((4, 4)); P[:2, :2] = C; P[2, 2] = P[3, 3] = 0.25 ** 2
        self.tracks.append({"id": f"FT{self.nid:03d}", "x": np.array([rep["x"], rep["y"], 0, 0]), "P": P, "k": k,
                            "scans": [k], "sensors": {rep["sensor"]: 1}, "truths": [rep["truth"]], "nis": [],
                            "hist": [(k, rep["x"], rep["y"])], "confirmed": False, "miss": 0, "born": k, "upd": True})
        self.nid += 1

    def _assign(self, tracks, rs, sid):
        """게이트(마할라노비스 거리) 안에서 전역 최근접 할당(GNN). 할당된 보고서 번호 집합을 돌려준다."""
        if not tracks or not rs:
            return set()
        spos = SENSORS[sid]["pos"]
        Z = np.array([[r["range_km"], np.radians(r["az_deg"])] for r in rs])
        R = np.diag([rs[0]["sr"] ** 2, np.radians(rs[0]["saz"]) ** 2])
        cost = np.full((len(tracks), len(rs)), 1e6)
        for i, tr in enumerate(tracks):
            zp, H = h_polar(tr["x"], spos)
            Y = Z - zp
            Y[:, 1] = (Y[:, 1] + np.pi) % (2 * np.pi) - np.pi
            Si = np.linalg.inv(H @ tr["P"] @ H.T + R)
            d2 = np.einsum("ij,jk,ik->i", Y, Si, Y)
            cost[i] = np.where(d2 < self.gate, d2, 1e6)
        used = set()
        for i, j in zip(*linear_sum_assignment(cost)):
            if cost[i, j] < self.gate:
                tr, r = tracks[i], rs[j]
                y, H, S, d2 = innovation(tr["x"], tr["P"], r)
                K = tr["P"] @ H.T @ np.linalg.inv(S)
                tr["x"] = tr["x"] + K @ y
                tr["P"] = (np.eye(4) - K @ H) @ tr["P"]
                tr["nis"].append(d2); tr["truths"].append(r["truth"])
                tr["sensors"][sid] = tr["sensors"].get(sid, 0) + 1
                tr["upd"] = True; used.add(j)
        return used

    def _merge(self):
        """같은 비행체를 따라가는 중복 항적(위치·속도가 통계적으로 같음)을 오래된 항적으로 통합한다."""
        drop = set()
        for a in range(len(self.tracks)):
            for b in range(a + 1, len(self.tracks)):
                A, B = self.tracks[a], self.tracks[b]
                if A["id"] in drop or B["id"] in drop:
                    continue
                d = A["x"] - B["x"]
                d2 = float(d[:2] @ np.linalg.solve(A["P"][:2, :2] + B["P"][:2, :2], d[:2]))
                if d2 < self.merge_gate and np.hypot(d[2], d[3]) < 0.05:
                    keep, lose = (A, B) if len(A["scans"]) >= len(B["scans"]) else (B, A)
                    for sid, n in lose["sensors"].items():
                        keep["sensors"][sid] = keep["sensors"].get(sid, 0) + n
                    keep["truths"] += lose["truths"]; keep["scans"] = sorted(set(keep["scans"]) | set(lose["scans"]))
                    keep["merged"] = keep.get("merged", 0) + 1
                    drop.add(lose["id"])
        self.tracks = [t for t in self.tracks if t["id"] not in drop]
        self.n_merged += len(drop)

    def step(self, k, reps):
        for tr in self.tracks:                                         # 1) 예측
            tr["x"], tr["P"] = ekf_predict(tr["x"], tr["P"], k - tr["k"], self.q)
            tr["k"], tr["upd"] = k, False
        for sid in sorted({r["sensor"] for r in reps}, key=lambda s: SENSORS[s]["sr"]):   # 2) 정밀 센서부터 순차 갱신
            rs = [r for r in reps if r["sensor"] == sid]
            used = self._assign([t for t in self.tracks if t["confirmed"]], rs, sid)      # 확정 항적 우선
            rest = [r for j, r in enumerate(rs) if j not in used]
            used2 = self._assign([t for t in self.tracks if not t["confirmed"]], rest, sid)
            for j, r in enumerate(rest):
                if j not in used2:
                    self._new(r, k)
        if self.merge_gate > 0:
            self._merge()
        keep = []
        for tr in self.tracks:                                         # 3) 확정·삭제 관리
            if tr["upd"]:
                tr["miss"] = 0
                if tr["scans"][-1] != k:
                    tr["scans"].append(k)
                tr["hist"].append((k, tr["x"][0], tr["x"][1]))
            else:
                tr["miss"] += 1
            m, n = self.confirm
            if not tr["confirmed"] and sum(1 for s in tr["scans"] if s > k - n) >= m:
                tr["confirmed"] = True
            limit = self.max_miss if tr["confirmed"] else 1
            (keep if tr["miss"] <= limit else self.dead).append(tr)
        self.tracks = keep

    def all_tracks(self):
        return self.dead + self.tracks


def run_tracker(reps, **kw):
    trk = Tracker(**kw)
    by = {}
    for r in reps:
        by.setdefault(int(round(r["t"])), []).append(r)
    for k in range(min(by), max(by) + 1):
        trk.step(k, by.get(k, []))
    return trk.all_tracks()


def label_track(tr):
    vals, cnt = np.unique(tr["truths"], return_counts=True)
    return str(vals[np.argmax(cnt)]), float(cnt.max() / cnt.sum())


def evaluate(tracks, scn, min_reports=5):
    """정답 연결 평가: 재현율, 항적 분할(한 표적에 항적 2개 이상), 거짓 항적, 병합 오류, 위치 오차."""
    conf = [t for t in tracks if t["confirmed"]]
    for t in conf:
        t["label"], t["purity"] = label_track(t)
    cnt = pd.Series([r["truth"] for r in scn["reports"] if r["truth"].startswith("TGT")]).value_counts()
    truths = [k for k, v in cnt.items() if v >= min_reports]
    per = {g: [t for t in conf if t["label"] == g] for g in truths}
    found = [g for g in truths if per[g]]
    split = sum(max(0, len(v) - 1) for v in per.values())
    false = [t for t in conf if not t["label"].startswith("TGT")]
    merge = [t for t in conf if t["label"].startswith("TGT") and t["purity"] < 0.7]
    err = []
    for g in found:
        for t in per[g]:
            for k, x, y in t["hist"][3:]:
                if 0 <= k < scn["T"]:
                    err.append(np.hypot(x - scn["targets"][g][k, 0], y - scn["targets"][g][k, 1]))
    n = len(truths)
    acc = len(found) / max(n + split + len(false) + len(merge), 1)
    return {"정답표적": n, "탐지표적": len(found), "재현율": round(len(found) / max(n, 1), 3), "분할": split,
            "거짓항적": len(false), "병합오류": len(merge), "위치RMSE(km)": round(float(np.sqrt(np.mean(np.square(err)))) if err else np.nan, 3),
            "항적정확도": round(acc, 3)}

INIT = dict(gate=5.99, q=5e-5, confirm=(2, 3), max_miss=1, merge_gate=0.0)          # 1차 배치 파라미터(가상)
AAR_SCN = [dict(seed=1001, special="turn", weather="rain"), dict(seed=1002, special="cross", fa_mult=2.0),
           dict(seed=1003, dropout=("RDR-A", 20, 35), weather="fog"), dict(seed=1004, special="fast", ghost=True)]


def run(kw, params, T=40):
    scn = make_scenario(T=T, **kw)
    trk = run_tracker(prepare(scn), **params)
    return evaluate(trk, scn), trk


def errors(trk):
    """오류 사례 목록: 분할(같은 표적 항적 2개 이상), 거짓 항적, 병합 오류."""
    conf = [t for t in trk if t["confirmed"]]
    out, seen = [], {}
    for t in sorted(conf, key=lambda t: t["born"]):
        if not t["label"].startswith("TGT"):
            out.append(("거짓항적", t["label"], len(t["sensors"]), len(t["scans"])))
        elif t["purity"] < 0.7:
            out.append(("병합오류", t["label"], len(t["sensors"]), len(t["scans"])))
        elif t["label"] in seen:
            out.append(("분할", t["label"], len(t["sensors"]), t["born"] - seen[t["label"]]))
        seen.setdefault(t["label"], t["born"])
    return out


print("[AAR 일지 분석 - 1차 배치 파라미터]", INIT)
log, acc0 = [], []
for i, kw in enumerate(AAR_SCN):
    ev, trk = run(kw, INIT)
    acc0.append(ev["항적정확도"])
    for e in errors(trk):
        log.append({"일지": f"AAR-{i + 1}", "조건": kw.get("special") or kw.get("weather") or "-", "오류": e[0], "표적": e[1], "센서수": e[2], "수명/간격(s)": e[3]})
A = pd.DataFrame(log)
print(f"시나리오 {len(AAR_SCN)}개 평균 항적 정확도 {np.mean(acc0):.1%}, 오류 {len(A)}건")
print(A.groupby(["일지", "조건", "오류"]).size().unstack(fill_value=0).to_string())
print("\n[원인 분석: 파라미터 하나씩만 바꿔 AAR 일지 재실행]  가설 -> 정확도 / 분할 / 거짓항적")
HYP = [("gate", 9.21, "게이트(5.99)가 좁아 기동·잡음 시 경보가 항적 밖으로 빠져 새 항적 생성"),
       ("q", 4e-4, "프로세스 잡음이 작아 선회·고속 기동 예측이 어긋남"),
       ("confirm", (4, 6), "확정 규칙(2/3)이 느슨해 클러터·기만 경보가 거짓 항적으로 확정"),
       ("max_miss", 4, "삭제 기준이 짧아 단절·미탐 1~2초에 항적이 끊겼다 재생성"),
       ("merge_gate", 6.0, "중복 통합 미사용으로 센서별 항적이 합쳐지지 않음")]
for k, v, why in HYP:
    evs = [run(kw, {**INIT, k: v})[0] for kw in AAR_SCN]
    acc = np.mean([e["항적정확도"] for e in evs])
    print(f" {k:10s}={str(v):8s} {acc:6.1%} / {sum(e['분할'] for e in evs):2d} / {sum(e['거짓항적'] for e in evs):2d}  ({acc - np.mean(acc0):+.1%}p) {why}")
print(" -> 정확도 개선 폭이 큰 파라미터가 주원인이다. 개선이 없거나 악화되는 가설은 기각하고 조합 탐색으로 최종값을 정한다.")

# 재보정: 별도 튜닝 시나리오에서 격자 탐색
TUNE = [dict(seed=1101, special="turn"), dict(seed=1102, fa_mult=2.0, weather="rain"), dict(seed=1103, dropout=("RDR-B", 10, 25), ghost=True)]
grid = list(itertools.product([9.21], [5e-5, 4e-4], [(2, 3), (3, 5), (4, 6)], [1, 4], [0.0, 6.0]))     # 게이트는 카이제곱 99%(9.21)로 고정
best = None
for g, q, c, mm, mg in grid:
    p = dict(gate=g, q=q, confirm=c, max_miss=mm, merge_gate=mg)
    a = np.mean([run(kw, p, T=30)[0]["항적정확도"] for kw in TUNE])
    if best is None or a > best[0]:
        best = (a, p)
tuned = best[1]
print(f"\n격자 탐색 {len(grid)}개 조합 -> 튜닝 시나리오 정확도 {best[0]:.1%}")
yaml = "# configs/ops02_tuned_params.yaml [교육용가상자료]\ntracker:\n" + "\n".join(f"  {k}: {list(v) if isinstance(v, tuple) else v}" for k, v in tuned.items())
yaml += "\n  sync: heartbeat_median\n  sensor_order: precise_first\nreview:\n  route_to_operator: [split_suspect, single_sensor_over_30s]"
print(yaml)

# 회귀 시험: AAR 시나리오 재발 + 새 시나리오 일반화
recur, fa_b, fa_a = 0, 0, 0
for kw in AAR_SCN:
    ev0, trk0 = run(kw, INIT)
    ev, trk = run(kw, tuned)
    old = {(e[0], e[1]) for e in errors(trk0) if e[0] != "거짓항적"}
    new = {(e[0], e[1]) for e in errors(trk) if e[0] != "거짓항적"}
    recur += len(old & new); fa_b += ev0["거짓항적"]; fa_a += ev["거짓항적"]
NEW = [dict(seed=1201 + i, **c) for i, c in enumerate([dict(special="turn"), dict(special="formation"), dict(weather="rain", fa_mult=2.0),
                                                       dict(dropout=("RDR-A", 10, 30)), dict(ghost=True, weather="night"), dict(special="fast")])]
b = [run(kw, INIT)[0]["항적정확도"] for kw in NEW]
a = [run(kw, tuned)[0]["항적정확도"] for kw in NEW]
print(f"\n[회귀 시험] 새 시나리오 {len(NEW)}개 항적 정확도: 재보정 전 {np.mean(b):.1%} -> 후 {np.mean(a):.1%} ({(np.mean(a) - np.mean(b)) * 100:+.1f}%p, 목표 +10%p: {'달성' if np.mean(a) - np.mean(b) >= 0.1 else '미달'})")
print(f"AAR 일지 오류 재발(같은 표적 분할·병합) {recur}건 (목표 0건: {'달성' if recur == 0 else '미달'}), 같은 일지 거짓 항적 {fa_b}건 -> {fa_a}건")
print("시나리오별 정확도(전->후):", [f"{x:.2f}->{y:.2f}" for x, y in zip(b, a)])
print("지속 학습 연계: 조작관 통합·분리 결정(미션 7 로그)을 정답 라벨로 누적 -> 분기별 재탐색 -> 본 회귀 시험 통과 시에만 파라미터 교체")

plt.figure(figsize=(6, 3.5))
x = np.arange(len(NEW))
plt.bar(x - 0.2, b, 0.4, label="initial"); plt.bar(x + 0.2, a, 0.4, label="retuned")
plt.xticks(x, ["turn", "formation", "rain+FA", "dropout", "ghost", "fast"]); plt.ylabel("track accuracy"); plt.legend(); plt.title("AAR recalibration (synthetic)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR 일지 분석 - 1차 배치 파라미터] {'gate': 5.99, 'q': 5e-05, 'confirm': (2, 3), 'max_miss': 1, 'merge_gate': 0.0}
시나리오 4개 평균 항적 정확도 33.8%, 오류 47건
오류           거짓항적  병합오류  분할
일지    조건                   
AAR-1 turn      6     1   5
AAR-2 cross     8     1   7
AAR-3 fog       4     0   5
AAR-4 fast      5     0   5

[원인 분석: 파라미터 하나씩만 바꿔 AAR 일지 재실행]  가설 -> 정확도 / 분할 / 거짓항적
 gate      =9.21      43.4% /  4 / 28  (+9.6%p) 게이트(5.99)가 좁아 기동·잡음 시 경보가 항적 밖으로 빠져 새 항적 생성
 q         =0.0004    35.8% / 21 / 22  (+2.0%p) 프로세스 잡음이 작아 선회·고속 기동 예측이 어긋남
 confirm   =(4, 6)    86.6% /  3 /  1  (+52.8%p) 확정 규칙(2/3)이 느슨해 클러터·기만 경보가 거짓 항적으로 확정
 max_miss  =4         34.8% / 23 / 22  (+1.0%p) 삭제 기준이 짧아 단절·미탐 1~2초에 항적이 끊겼다 재생성
 merge_gate=6.0       32.1% / 17 / 36  (-1.7%p) 중복 통합 미사용으로 센서별 항적이 합쳐지지 않음
 -> 정확도 개선 폭이 큰 파라미터가 주원인이다. 개선이 없거나 악화되는 가설은 기각하고 조합 탐색으로 최종값을 정한다.

격자 탐색 24개 조합 -> 튜닝 시나리오 정확도 95.2%
# configs/ops02_tuned_params.yaml [교육용가상자료]
tracker:
  gate: 9.21
  q: 5e-05
  confirm: [4, 6]
  max_miss: 1
  merge_gate: 0.0
  sync: heartbeat_median
  sensor_order: precise_first
review:
  route_to_operator: [split_suspect, single_sensor_over_30s]

[회귀 시험] 새 시나리오 6개 항적 정확도: 재보정 전 29.7% -> 후 92.1% (+62.4%p, 목표 +10%p: 달성)
AAR 일지 오류 재발(같은 표적 분할·병합) 0건 (목표 0건: 달성), 같은 일지 거짓 항적 23건 -> 1건
시나리오별 정확도(전->후): ['0.33->1.00', '0.35->1.00', '0.13->1.00', '0.32->1.00', '0.25->0.67', '0.40->0.86']
지속 학습 연계: 조작관 통합·분리 결정(미션 7 로그)을 정답 라벨로 누적 -> 분기별 재탐색 -> 본 회귀 시험 통과 시에만 파라미터 교체
```
</details>
